# 03 - Baseline Forecasts & Error Metrics

**What a baseline is:** the simplest reasonable forecasting rule you can write down — no model training, just arithmetic on history. You build one *before* the real model for one reason: without it, you have no way to know whether a fancier model is actually earning its complexity. If a tree-based model can't beat "average of the last 4 same weekdays," it isn't worth using.

With data now spanning close to 2 years, we can finally build **four** baselines instead of two — including the classic "last year, same date" rule notebook 03 originally had to skip:

1. **Naive lag-7** — forecast = covers on the same weekday, one week ago.
2. **4-week rolling average per weekday** — forecast = average of the last 4 occurrences of that weekday.
3. **Same calendar date, last year** — forecast = covers on this exact date 365 days ago. The classic restaurant baseline, now finally testable.
4. **Same weekday, ~52 weeks ago** — a small fix to #3, explained once you see why #3 underperforms.

## Step 1: Load data and set up a time-based train/test split

**Why time-based, not random:** a forecasting model's whole job is predicting the *future* from the *past*. If you randomly shuffle days into train/test, the model can end up training on a Tuesday from next month and being tested on a Tuesday from last month — it's effectively allowed to peek at the future. That's **data leakage**, and it makes your error metrics lie to you (too optimistic). The fix is simple: pick a cutoff date, everything before it is train, everything on/after it is test, and never let information from a test-period date influence a train-period prediction.

We'll hold out the **last 8 weeks** as the test period and use everything before that as train/history.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/covers_daily_clean.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df[df["is_closed"] == 0].reset_index(drop=True)  # closed days aren't a forecasting target

test_start = df["date"].max() - pd.Timedelta(days=55)  # last 8 weeks
train = df[df["date"] < test_start].copy()
test = df[df["date"] >= test_start].copy()

print(f"Train: {train['date'].min().date()} to {train['date'].max().date()} ({len(train)} days)")
print(f"Test:  {test['date'].min().date()} to {test['date'].max().date()} ({len(test)} days)")

## Step 2: Baseline A — naive lag-7

For every date in the test period, the forecast is just the actual covers from exactly 7 days earlier. Because it's always looking exactly one week back, this rule never needs a separate "training step" — it works directly off the full history, and it's automatically walk-forward (each prediction only ever uses data that's already happened by that point).

In [ ]:
covers_by_date = df.set_index("date")["covers"]

lag7_forecast = covers_by_date.shift(7)  # value from 7 days before each date
test = test.set_index("date")
test["pred_lag7"] = lag7_forecast.reindex(test.index)
test = test.reset_index()

test[["date", "day_of_week", "covers", "pred_lag7"]].head(10)

## Step 3: Baseline B — 4-week rolling average per weekday

For each test date, look back through history for the **same weekday** and average the 4 most recent occurrences *before* that date. This has to be computed walk-forward, one test date at a time, using only data available up to the day before — otherwise a June prediction could accidentally use a July value, which is the leakage problem from Step 1 again.

In [ ]:
def rolling_weekday_avg(target_date, dow, history_df, n=4):
    """Average covers from the n most recent same-weekday rows strictly before target_date."""
    past_same_dow = history_df[(history_df["date"] < target_date) & (history_df["day_of_week"] == dow)]
    recent = past_same_dow.sort_values("date").tail(n)
    return recent["covers"].mean() if len(recent) > 0 else np.nan

test["pred_roll4"] = [
    rolling_weekday_avg(row.date, row.day_of_week, df)
    for row in test.itertuples()
]

test[["date", "day_of_week", "covers", "pred_lag7", "pred_roll4"]].head(10)

## Step 4: Baseline C — same calendar date, last year

The forecast a lot of restaurant folks reach for first: "what did we do on this exact date last year?" It needs a full year of prior history to work at all, which is exactly why notebook 03 originally couldn't build it — the data started less than 12 months before the test period. It can now.

In [ ]:
test["pred_lastyear_date"] = test["date"].apply(lambda d: covers_by_date.get(d - pd.Timedelta(days=365), np.nan))

test[["date", "day_of_week", "covers", "pred_lastyear_date"]].head(10)

**Before scoring it, check something easy to miss:** what day of the week was "365 days ago"?

In [ ]:
sample_date = test["date"].iloc[0]
one_year_back = sample_date - pd.Timedelta(days=365)
print(f"{sample_date.date()} is a {sample_date.day_name()}")
print(f"{one_year_back.date()}, 365 days earlier, is a {one_year_back.day_name()}")

**365 isn't divisible by 7.** Every non-leap year shifts the calendar by one weekday, so "the same date last year" quietly lands on a *different* day of the week roughly every year — here, comparing a quiet Monday to what was actually a busy Sunday twelve months earlier. Combined with this restaurant's overall growth over the two years (notebook 02's year-over-year table), that's two separate reasons this baseline could run badly wrong, not one. Score it before deciding — this is exactly the kind of assumption worth checking with numbers instead of intuition.

## Step 5: Baseline D — same weekday, ~52 weeks ago (the fix)

Same idea as Baseline C, but anchored on **364 days** back (exactly 52×7) instead of 365. That keeps the weekday aligned, at the cost of landing one calendar day off from the literal anniversary date — a trade worth making, given what Step 4 just found.

In [ ]:
test["pred_lastyear_weekday"] = test["date"].apply(lambda d: covers_by_date.get(d - pd.Timedelta(days=364), np.nan))

# Confirm the fix actually holds the weekday constant
check_date = test["date"].iloc[0] - pd.Timedelta(days=364)
print(f"364 days back from {test['date'].iloc[0].date()} is {check_date.date()}, a {check_date.day_name()} — matches.")

test[["date", "day_of_week", "covers", "pred_lastyear_date", "pred_lastyear_weekday"]].head(10)

## Step 6: Score all four baselines with MAE and MAPE

In [ ]:
def mae(pred, actual):
    return (pred - actual).abs().mean()

def mape(pred, actual):
    return ((pred - actual).abs() / actual).mean() * 100

results = []
for name, col in [
    ("Naive lag-7", "pred_lag7"),
    ("4-week rolling avg (weekday)", "pred_roll4"),
    ("Same date, last year", "pred_lastyear_date"),
    ("Same weekday, ~52 weeks ago", "pred_lastyear_weekday"),
]:
    valid = test[col].notna()
    results.append({
        "baseline": name,
        "MAE": round(mae(test.loc[valid, col], test.loc[valid, "covers"]), 1),
        "MAPE (%)": round(mape(test.loc[valid, col], test.loc[valid, "covers"]), 1),
        "n_days": int(valid.sum()),
    })

results_df = pd.DataFrame(results).sort_values("MAE")
results_df

**Read this table as:** the lowest MAE/MAPE combination is the number the real model (notebook 05) has to beat. The literal "same date last year" typically comes out clearly *worst* here — a good reminder that a classic rule of thumb isn't automatically the right one for a specific restaurant's data, and that it's worth testing rather than assuming. The weekday-aligned fix (Baseline D) closes most of that gap, but a single data point from a year ago is still noisier than averaging several recent occurrences — which is why the **4-week rolling average usually still wins**: recent data, smoothed, beats old data, even when the old data is looking at the right day of the week.

## Step 7: Visualize actual vs. all four baselines


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(test["date"], test["covers"], label="Actual", color="black", linewidth=1.8)
ax.plot(test["date"], test["pred_lag7"], label="Naive lag-7", alpha=0.6, linestyle="--")
ax.plot(test["date"], test["pred_roll4"], label="4-week rolling avg", alpha=0.9, linestyle="--")
ax.plot(test["date"], test["pred_lastyear_weekday"], label="Same weekday, ~52 wks ago", alpha=0.6, linestyle=":")
ax.set_title("Test period: actual covers vs. baselines (literal same-date-last-year omitted \u2014 too noisy to read on this scale)")
ax.set_xlabel("Date")
ax.set_ylabel("Covers")
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Step 8: Where do the baselines struggle?

Break the error down by weekday, then look at the single worst-predicted days in detail. This tells you *where* a smarter model needs to add the most value — not just that it should be better on average.

In [ ]:
test["err_lag7"] = (test["pred_lag7"] - test["covers"]).abs()
test["err_roll4"] = (test["pred_roll4"] - test["covers"]).abs()
test["err_lastyear_date"] = (test["pred_lastyear_date"] - test["covers"]).abs()
test["err_lastyear_weekday"] = (test["pred_lastyear_weekday"] - test["covers"]).abs()

by_weekday = test.groupby("day_of_week")[["err_lag7", "err_roll4", "err_lastyear_date", "err_lastyear_weekday"]].mean().round(1)
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday.reindex(dow_order)

In [ ]:
worst = test.sort_values("err_roll4", ascending=False).head(8)
worst[["date", "day_of_week", "covers", "pred_roll4", "err_roll4",
       "is_holiday", "holiday_name", "is_local_event", "event_name",
       "is_promo", "condition", "notes"]]

**Read the worst-day table.** In this run, the biggest misses cluster around exactly the things neither baseline can see: a Farmers Market Expansion event driving a Thursday spike, a July 4th holiday landing on a rainy Saturday, and a one-off midday closure for a health inspection. Both baselines only know "what day of the week is it" — they have no way to use `is_holiday`, `is_local_event`, `is_promo`, or weather, even though notebook 02 showed those all move covers. That gap is exactly what the real model in notebook 05 is for.

## Recap

- Built a **time-based train/test split** (last 8 weeks held out) instead of a random split, to avoid leaking future information into training.
- **Naive lag-7** (same weekday, one week ago) — simplest baseline, easy to compute.
- **4-week rolling average per weekday** smooths out single-week noise, and comes out **best of all four** baselines on both MAE and MAPE in this dataset.
- **Same date, last year** — now testable with ~2 years of history, and it's a real lesson, not just a formality: 365 isn't divisible by 7, so "the same date" quietly lands on a different weekday most years, which combined with this restaurant's growth over time made it the *worst* of the four baselines here.
- **Same weekday, ~52 weeks ago** — a one-line fix (anchor on 364 days instead of 365) that closes most of that gap, though a single year-old data point is still noisier than averaging several recent ones.
- The worst-predicted days line up with holidays, local events, and one-off closures — signals every one of these baselines is blind to, which is exactly the gap a feature-based model can close.

**Next up:** feature engineering — turning the day-of-week, rolling-average, and event/holiday/promo ideas from notebooks 02 and 03 into proper model inputs (lags, rolling windows, binary flags), then fitting a real model and seeing whether it beats the 4-week rolling average baseline.